# Preparation

<!-- structured-notebook -->
## Notebook Summary
Purpose: run several rounds of manual Guardian corpus cleaning, alternating between tag cleanup, BERTopic re-fitting, and per-topic keep/drop decisions until a usable cleaned corpus is produced.

Main steps:
- Start from a previously cleaned Guardian export and inspect the initial topic structure.
- Iteratively rerun BERTopic and document manual decisions for noisy or mixed topics.
- Save intermediate cleaned versions so later rounds can continue from a more focused corpus.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.news.nlp import *
from itables import show
import itables.options as opt

opt.maxBytes = 0  # Disabling size limit for big tables

/home/iskender/anaconda3/envs/bertopic-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Analysis

### Tag Cleaning Round 1

`import re

keywords = ["ageing", "life", "health", "wellbeing","nutrition", "aging", "longevity", "healthy aging", "anti-aging", "anti ageing"]
<br>
pattern = "|".join([kw.replace(" ", r"\s+") for kw in keywords])

mask = guardian['tags'].str.contains(pattern, case=False, na=False)
<br>
guardian_age = guardian[mask].reset_index(drop=True)

print(f"Kept {len(guardian_age)} of {len(guardian)} articles related to ageing.")
***
Kept 1303 of 5498 articles related to ageing.
***
guardian_age.to_csv('../GuardianAPI/guardian_cleared1.csv')`

<!-- structured-notebook -->
## Round 2 Overview
This pass reruns BERTopic after the initial tag cleanup. The following topic subsections should be read as a manual decision log: inspect the cluster, decide what to keep, and update the working dataframe.


### BERTopic Cleaning round 2

In [2]:
full_keyword_getter(guardian_out)

NameError: name 'guardian_out' is not defined

#### Cleaning topic-0

In [23]:
topic_0 = guardian_out[guardian_out['topic'] == 0]

In [25]:
model, topic_info, topic0_out = analyze_bertopic(topic_0, v_min_df=1, v_max_df=0.85,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

2025-07-27 13:20:53,004 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████████████████████████████| 15/15 [02:03<00:00,  8.21s/it]
2025-07-27 13:22:56,323 - BERTopic - Embedding - Completed ✓
2025-07-27 13:22:56,323 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-07-27 13:23:01,056 - BERTopic - Dimensionality - Completed ✓
2025-07-27 13:23:01,057 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-07-27 13:23:01,077 - BERTopic - Cluster - Completed ✓
2025-07-27 13:23:01,078 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-07-27 13:23:01,732 - BERTopic - Representation - Completed ✓
2025-07-27 13:23:01,733 - BERTopic - Topic reduction - Reducing number of topics
2025-07-27 13:23:01,737 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-07-27 13:23:02,367 - BERTopic - Representation - Completed ✓
2025-07-27 13:23:02,368 - BERTopic 

In [26]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8,-1_onset dementia_daughter_early onset_cyclists,"[onset dementia, daughter, early onset, cyclis...",[That conciliatory question mark is unlikely t...
1,0,173,0_cells_mice_humans_scientists,"[cells, mice, humans, scientists, oldest, drug...",[Tech entrepreneur Bryan Johnson is talking ab...
2,1,142,1_pension_older people_state pension_pensions,"[pension, older people, state pension, pension...",[Younger generations are being told to prepare...
3,2,65,2_muscle_yoga_gym_workout,"[muscle, yoga, gym, workout, strength, minutes...",[When Baz Luhrmann called the body “the greate...
4,3,55,3_abortion_political_amendment_mps,"[abortion, political, amendment, mps, vote, se...","[“I worry about everything,” the prime ministe..."
5,4,14,4_dying_funeral_nazi_ss,"[dying, funeral, nazi, ss, trauma, truth, mcca...",[We are often told that in earlier times all c...
6,5,12,5_sleep_brain health_screening_70s,"[sleep, brain health, screening, 70s, screenin...",[People in their 50s can be prone to a “viciou...


In [30]:
full_keyword_getter(topic0_out)

-1.0 → ['onset dementia', 'daughter', 'early onset', 'cyclists', 'socioeconomic', 'royal mail', 'mail', 'bikes', 'royal', 'socioeconomic status', 'raf', 'military', 'drivers', 'defence', 'cycling', 'cyclist', 'decision', 'sentence', 'risk developing', 'traffic', 'end life', 'lifestyle factors', 'channel', 'sales', 'delivery', 'healthy lifestyle', 'lord', 'road', 'campaign', 'tells']
0.0 → ['cells', 'mice', 'humans', 'scientists', 'oldest', 'drug', 'diet', 'diseases', 'cell', 'average', 'anti', 'drugs', 'centenarians', 'japan', 'company', 'senescent', 'genetic', 'genes', 'johnson', 'anti ageing', 'scientific', 'senescent cells', 'trial', 'birth', 'stem', 'animals', 'plasma', 'treatments', 'levels', 'dna']
1.0 → ['pension', 'older people', 'state pension', 'pensions', 'income', 'average', 'pension age', 'job', 'workers', 'jobs', 'areas', 'housing', 'paid', 'calment', 'generation', 'ageism', 'insurance', 'poverty', 'social care', 'career', 'rise', 'zak', 'skills', 'north', 'city', 'inequa

##### Cleaning subtopics

###### Cleaning subtopic-5

In [31]:
topic0_5 = topic0_out[topic0_out['topic'] == 5]

In [33]:
show(topic0_5[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
The topics here are quite related to longevity, we leave them be.
***

###### Cleaning subtopic-4

In [35]:
topic0_4 = topic0_out[topic0_out['topic'] == 4]
show(topic0_4[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
None of topics here except 242 are related to longevity, so let's leave them out.
***

In [36]:
indices_to_remove = topic0_4.index.difference([242])

guardian_out = guardian_out.drop(indices_to_remove)

#### Cleaning topic-11

***
['wines', 'vodka', 'mid range', 'sparkling', 'splurge', 'price range', 'madeira', 'flavours', 'socks', 'reds', 'jeans'<br>
 'cocktail', 'craft', 'red wine', 'grape', 'linen', 'grapes', 'olive oil', 'shirt', 'bedding','citrus', 'pint', 'cocktails',<br> 
 'producers', 'high quality', 'moisturiser', 'durability', 'premium', 'syrup', 'portugal']
 ***

In [2]:
topic_11 = guardian_out[guardian_out['topic'] == 11]

In [3]:
pd.set_option('display.max_colwidth', None)
topic_11[['headline','trailText','bodyText','links', 'tags']]

headline  \
169                                                                                              Bottles with a seal of approval   
311   ‘If you pay more than £4, you’re being ripped off’: the fair price for 14 everyday items, from cleaning spray to olive oil   
435                                                                                         The magic of vintage fortified wines   
444                                                             The marvels of Madeira: one of the great unsung Portuguese wines   
612                                                           Sick of spritzes? The ultimate guide to Australian summer drinking   
623                                                                                                How malbec scaled the heights   
665                                                               French family's last three bottles of 1774 wine up for auction   
794                                                    England’s vineyards meet their biggest challenge – a decent bottle of red   
873                                                          From amari mania to a vodka revival: what we’ll be drinking in 2017   
967                                                                 Co-operative Group toasts success of world-beating champagne   
1137                                                             Drink up, it's good for you! A red wine detox holiday in France   

                                                                                                                                                                                                   trailText  \
169                                                                                                         Wines that benefit from the influence of a new generation of master-craftspeople in the vineyard   
311   Everything seems so expensive these days – but how much should you pay for that bottle of wine, or pair of jeans? A panel of experts explains how to make good choices – and when you’re being fleeced   
435                                                                              Complex and beguiling, very old port, sherry and madeira can pack an emotional – and financial – punch, says David Williams   
444                                                                                      Madeira may be a great and versatile cooking ingredient, but it’s also the store-cupboard tipple par excellence<br>   
612                                               From refreshing cocktails to savoury, grown-up non-alcoholic options, this summer, drinking is lower on sugar, higher on flavour and a little experimental   
623                                                                                                                              The rise of the grape that’s at its best on mountain slopes is an epic tale   
665                                                                                                            Other bottles of same wine – among the oldest in the world – previously fetched up to €57,000   
794                                                                                 The UK climate has long cast a shadow over a homegrown pinot noir, but after years of experiments, is this now changing?   
873                                                                                     If you want to keep up with the in-things in drinks this year, here are four trends that could take off in a big way   
967                                                                                      <p>Cut-price Les Pionniers vintage champagne wows judges to scoop three of industry's top international honours</p>   
1137                                     <p>You don't usually associate detoxing with fun – or booze. But there's plenty of both on a new red wine detox in the Madiran wine region in south-west France</p>   

                                      

***
None of the topics are related to healthy ageing so we get rid of them.
***

In [13]:
guardian_out = guardian_out[guardian_out['topic']!=11]

#### Cleaning topic-10

***
['productivity', 'insomnia', 'beds', 'caffeine', 'nap', 'hours sleep', 'boredom', 'bedtime', 'cognitive',<br>
'awake', 'seven hours', 'coach', 'bored', 'sleep night', 'working home', 'saunders', 'night sleep', 'microdosing',<br>
'waking', 'dopamine', 'physical activity', 'scrolling', 'neuroscience', 'duration', 'hormone', 'mindfulness',<br>
'creativity', 'time day', 'cup coffee', 'study suggests']
 ***

In [5]:
topic_10 = guardian_out[guardian_out['topic'] == 10]

In [6]:
topic_10

id  \
87                         lifeandstyle/article/2024/jun/23/sleep-restriction-insomnia-lara-williams-treats-supper-club-odyssey-cbti   
108   lifeandstyle/2025/jul/14/terrible-nights-sleep-heres-how-to-make-it-through-the-day-and-maybe-even-enjoy-it-one-step-at-a-time   
173                                                            lifeandstyle/2024/oct/18/improve-focus-concentration-span-expert-tips   
254                                                        world/2023/jun/23/in-the-home-of-the-siesta-the-nap-is-just-a-dream-spain   
423                                         lifeandstyle/2021/jan/01/burnt-out-resolutions-a-feel-better-guide-for-the-already-tired   
537                             lifeandstyle/2020/sep/24/the-golden-hour-15-ways-to-change-your-life-in-a-work-from-home-lunch-break   
584                                                 books/2019/aug/16/rolled-over-why-did-married-couples-stop-sleeping-in-twin-beds   
633                                               science/2018/may/23/weekend-lie-ins-could-help-you-avoid-an-early-death-study-says   
719                                           wellbeing-at-work/2017/sep/20/do-yoga-work-harder-how-productivity-co-opted-relaxation   
734                  lifeandstyle/2018/sep/03/sad-summers-over-18-ways-to-keep-the-health-humour-and-happiness-of-your-holiday-alive   
866                                                   lifeandstyle/2016/feb/15/is-it-good-to-be-bored-or-will-it-be-the-death-of-you   
1144                                                                     commentisfree/2012/nov/05/live-long-lazy-britain-oldest-man   
1252                                                           lifeandstyle/2010/jun/12/indigestion-social-events-hours-sleep-needed   

               section              pub_date  \
87      Life and style  2024-06-23T09:00:09Z   
108     Life and style  2025-07-14T04:00:02Z   
173     Life and style  2024-10-18T10:00:22Z   
254         World news  2023-06-23T10:59:06Z   
423     Life and style  2020-12-31T16:30:46Z   
537     Life and style  2020-09-24T10:30:43Z   
584              Books  2019-08-16T14:25:09Z   
633            Science  2018-05-23T03:00:11Z   
719   Guardian Careers  2017-09-20T15:16:45Z   
734     Life and style  2018-09-03T05:00:43Z   
866     Life and style  2016-02-15T07:45:04Z   
1144           Opinion  2012-11-05T18:40:08Z   
1252    Life and style  2010-06-11T23:01:29Z   

                                                                                                                                                         web_url  \
87                         https://www.theguardian.com/lifeandstyle/article/2024/jun/23/sleep-restriction-insomnia-lara-williams-treats-supper-club-odyssey-cbti   
108   https://www.theguardian.com/lifeandstyle/2025/jul/14/terrible-nights-sleep-heres-how-to-make-it-through-the-day-and-maybe-even-enjoy-it-one-step-at-a-time   
173                                                            https://www.theguardian.com/lifeandstyle/2024/oct/18/improve-focus-concentration-span-expert-tips   
254                                                        https://www.theguardian.com/world/2023/jun/23/in-the-home-of-the-siesta-the-nap-is-just-a-dream-spain   
423                                         https://www.theguardian.com/lifeandstyle/2021/jan/01/burnt-out-resolutions-a-feel-better-guide-for-the-already-tired   
537                             https://www.theguardian.com/lifeandstyle/2020/sep/24/the-golden-hour-15-ways-to-change-your-life-in-a-work-from-home-lunch-break   
584                                                 https://www.theguardian.com/books/2019/aug/16/rolled-over-why-did-married-couples-stop-sleeping-in-twin-beds   
633                                               https://www.theguardian.com/science/2018/may/23/weekend-lie-ins-could-help-you-avoid-an-early-death-study-says   
719                                           https://www.theguardian.com/we

***
None of these topics are related to longevity except these 2:
<br>science/2018/may/23/weekend-lie-ins-could-help-you-avoid-an-early-death-study-says
<br>lifeandstyle/2016/feb/15/is-it-good-to-be-bored-or-will-it-be-the-death-of-you
***

In [14]:
guardian_out = guardian_out[
    (guardian_out['topic'] != 10) |
    (
        (guardian_out['id'] == 'science/2018/may/23/weekend-lie-ins-could-help-you-avoid-an-early-death-study-says') |
        (guardian_out['id'] == 'lifeandstyle/2016/feb/15/is-it-good-to-be-bored-or-will-it-be-the-death-of-you')
    )
]

#### Cleaning topic-9

***
['chemotherapy', 'screening', 'oncologist', 'prostate', 'diagnosis', 'remission', 'prostate cancer', <br>
 'long covid', 'breast cancer', 'nursing', 'syphilis', 'quality life', 'toxicity', 'neurologist', <br>
 'cancer patients', 'surgeon', 'eczema', 'nursing home', 'cancers', 'palliative', 'complications', <br>
 'organ', 'mom', 'transplant', 'spine', 'craig', 'kidney', 'relieved', 'cognitive', 'illnesses']
***

In [8]:
topic_9 = guardian_out[guardian_out['topic'] == 9]

In [9]:
topic_9[['id','section','headline','trailText', 'links','tags']]

id  \
179                                           lifeandstyle/2023/apr/23/the-moment-i-knew-his-accident-in-a-weird-way-was-a-renewal-of-our-vows   
186                                                                                                 society/2024/may/08/kris-hallenga-obituary   
192                                                                society/2024/feb/22/fighting-talk-why-we-should-mind-our-language-on-cancer   
228                                                      commentisfree/2024/feb/16/king-charles-speak-cancer-winning-losing-war-bowel-language   
262                                                                      science/2023/jul/09/microbiome-chronic-fatigue-me-long-covid-research   
297                                             commentisfree/2023/jun/21/i-am-an-oncologist-can-chatgpt-help-me-deliver-bad-news-to-a-patient   
356                       commentisfree/2022/nov/02/its-not-looking-good-is-it-why-patients-families-plead-for-connection-as-much-as-prognosis   
374                     commentisfree/2022/sep/01/three-things-i-have-learned-about-end-of-life-care-from-treating-elderly-couples-with-cancer   
379   commentisfree/2022/aug/03/when-a-patients-survival-is-dwarfed-by-the-logistics-of-treatment-oncologists-need-to-talk-about-time-toxicity   
513                  society/2021/may/19/kris-hallenga-the-woman-diagnosed-with-cancer-at-23-who-convinced-a-generation-to-check-their-breasts   
531               commentisfree/2020/jun/23/i-understand-why-my-patient-is-scared-to-lose-her-hair-it-signals-the-loss-of-life-as-she-knows-it   
593              commentisfree/2019/sep/18/how-can-i-tell-my-patients-theyll-be-fine-in-a-nursing-home-our-trust-in-the-industry-was-misplaced   
754                                                   society/2018/jan/24/to-treat-or-not-to-treat-find-out-what-really-matters-to-the-patient   
774                                 commentisfree/2017/aug/30/as-cancer-progresses-some-patients-weep-some-get-angry-and-others-are-bewildered   
820                                                                         society/2016/sep/14/monitoring-prostate-cancer-effective-treatment   
838                                                                 healthcare-network/2016/sep/22/dying-patient-taught-me-doing-nothing-brave   
939                                                                           society/2015/jul/25/morquio-syndrome-nhs-drug-funding-sarah-long   
949                                                 science/sifting-the-evidence/2015/jul/06/why-are-some-diseases-screened-for-but-not-others   
1001                                                                     commentisfree/2015/mar/16/die-words-daughter-cancer-time-neurosurgeon   
1063                                                                                          society/2014/feb/11/ration-cancer-treatments-nhs   
1092                                                                           science/2014/nov/14/-sp-so-youve-survived-cancer-but-whats-next   
1110                                                                                society/2013/apr/09/families-reduce-organ-donation-refusal   
1117                                                                            society/2012/jul/21/mother-dementia-care-elderly-michael-wolff   
1119                                                                                       lifeandstyle/2012/nov/18/mother-lung-cancer-smoking   
1128                                                                      commentisfree/2013/jan/30/response-stigma-britten-death-still-potent   
1296                                                                                               lifeandstyle/2010/jan/12/living-with-eczema   

                               section  \
179                     Life and style   
186                            Society   
192                            Society   
228                          

***
None of the articles here are directly related to healthy ageing so we delete them.
***

In [15]:
guardian_out = guardian_out[guardian_out['topic'] != 9]

#### Cleaning topic-8

***
['flower', 'butterflies', 'roses', 'planting', 'foliage', 'blooms', 'flowering',<br>
'orchard', 'butterfly', 'hirst', 'cuttings', 'shrubs', 'compost', 'bulbs', 'iris',<br>
'edible', 'varieties', 'woodland', 'stems', 'oak', 'orchards', 'watering', 'shrub',<br>
'planted', 'chelsea', 'potter', 'container', 'bloom', 'bailey', 'potatoes']
***

In [11]:
topic_8 = guardian_out[guardian_out['topic']==8]

In [15]:
topic_8[['id','section','headline','trailText', 'links','tags']]

,id,section,headline,trailText,links,tags
31,lifeandstyle/2025/jul/11/foxgloves-cottage-garden-classics-town-urban-garden,Life and style,"Foxgloves are cottage garden classics, but they look just as good in town",These quintessential country flowers relish the dappled shade often cast in built-up areas – and there’s a variety for everyone,https://www.rhs.org.uk/plants/150196/digitalis-mertonensis-summer-king/details; https://www.rhs.org.uk/plants/150196/digitalis-mertonensis-summer-king/details; https://www.rhs.org.uk/plants/150196/digitalis-mertonensis-summer-king/details; https://www.rhs.org.uk/plants/339227/digitalis-martina/details; https://www.rhs.org.uk/plants/339227/digitalis-martina/details; https://www.rhs.org.uk/plants/339227/digitalis-martina/details; https://www.rhs.org.uk/plants/96171/digitalis-purpurea-sutton-s-apricot/details; https://www.rhs.org.uk/plants/136477/digitalis-purpurea-pam-s-choice/details; https://www.rhs.org.uk/plants/41734/digitalis-ferruginea/details; https://www.rhs.org.uk/plants/41734/digitalis-ferruginea/details; https://www.rhs.org.uk/plants/41734/digitalis-ferruginea/details; https://www.rhs.org.uk/plants/5861/digitalis-parviflora-jacq/details,Gardening advice; Gardens; Life and style
71,lifeandstyle/2025/may/09/mothers-day-2025-best-flowers-how-to-order-delivery,Life and style,Avoid same-day wilt: how to order the freshest Mother’s Day flowers,"It is estimated only 50% of flowers sold in Australia are locally grown, but florists and growers say local is ‘a superior product’. Here they share tips for buying blooms that last",https://www.fairtrade.net/content/fairtrade/language-masters/en/products/Flowers.html; https://www.theguardian.com/newsletters/2019/oct/18/saved-for-later-sign-up-for-guardian-australias-culture-and-lifestyle-email?CMP=copyembed; https://www.theguardian.com/lifeandstyle/2025/may/06/im-losing-my-mum-to-young-onset-dementia-caring-for-my-baby-reminds-me-who-she-was,Australian lifestyle; Life and style; Plants; Mother's Day
128,lifeandstyle/2025/apr/03/the-leaves-fall-off-but-i-think-thats-normal-the-houseplants-you-just-cant-kill,Life and style,‘The leaves fall off – but I think that’s normal’: the houseplants you just can’t kill,"Some indoor plants wither the moment you turn your back; others shrug off drought, darkness and even ‘watering’ by cats. Here’s how to choose the most hardy specimens. Plus, readers celebrate the greenery that survived against all the odds","https://www.theguardian.com/profile/guardian-community-team; https://www.theguardian.com/tone/letters; mailto:guardian.letters@theguardian.com?body=Please%20include%20your%20name%E2%80%8B%E2%80%8B,%20full%20postal%20address%20and%20phone%20number%20with%20your%20letter%20below.%20Letters%20are%20usually%20published%20with%20the%20author%27s%20name%20and%20city/town/village.%20The%20rest%20of%20the%20information%20is%20for%20verification%20only%20and%20to%20contact%20you%20where%20necessary.",Houseplants; Life and style; Interiors; Gardening advice; Homes
181,lifeandstyle/2024/jan/26/winter-is-not-for-chilling-its-time-to-sharpen-your-tools-and-prep-the-garden,Life and style,Winter is not for chilling – it’s time to sharpen your tools and prep the garden,"It may not be tempting to head outside in the cold, but there’s plenty to be getting on with – and you’ll thank yourself later in the year",https://www.niwaki.com/crean-mate/; https://www.niwaki.com/videos/,Gardening advice; Gardens; Life and style
184,lifeandstyle/2023/nov/03/houseplant-of-the-week-sea-urchin-cactus,Life and style,Houseplant of the week: sea urchin cactus,"Compact, non-spiny and with large luscious flowers, this Tex-Mex native will suit any home",NaN,Houseplants; Life and style
195,lifeandstyle/article/2024/may/03/10-shrubs-that-will-thrive-in-your-garden,Life and style,Shrubs offer resilience and reliability: here are 10 that will thrive in your garden,"As climate change brings volatile weather patterns, the shrub – too often ignored in fa

***
All of the articles are closer to gardening than to healthy ageing.
***

In [16]:
guardian_out = guardian_out[guardian_out['topic']!=8]

#### Cleaning topic-7

***
['nsw', 'covid 19', 'labor', 'vaccine', 'australians', 'queensland', 'coronavirus',<br>
'federal', 'court', 'premier', 'reports', 'election', 'abc', 'yesterday', 'liberal',<br>
'israel', 'dawson', 'referendum', 'virus', 'coalition', 'tuesday', 'vaccination',<br>
'vaccinated', 'variant', 'indigenous', 'opposition', 'quarantine', 'morrison', 'inquiry', 'hancock']
***

In [19]:
topic_7 = guardian_out[guardian_out['topic']==7]

In [20]:
topic_7[['id','section','headline','trailText', 'links','tags']]

id  \
207                                                    theobserver/commentisfree/2024/apr/28/learn-lessons-covid-before-another-deadly-disease-strikes-observer-letters   
222                                                                                                           us-news/2024/jan/15/covid-flu-rsv-mask-policies-hospitals   
293           australia-news/live/2025/apr/25/australia-election-2025-live-anthony-albanese-peter-dutton-anzac-day-dawn-ceremony-campaign-labor-hastie-coalition-ntwnfb   
310                                                      australia-news/2023/may/23/end-of-native-logging-in-victoria-a-monumental-win-for-forests-say-conservationists   
315                                               australia-news/2023/mar/30/covid-vaccine-booster-doses-only-needed-for-high-risk-groups-who-world-health-organization   
334                                                                   society/2022/aug/31/most-extraordinary-geoffrey-cumming-wows-melbourne-with-250m-medical-donation   
345                                           australia-news/2021/jul/17/secret-weapon-unflappable-kerry-chant-faces-toughest-challenge-yet-as-nsw-covid-outbreak-grows   
346       australia-news/live/2025/may/14/australia-news-live-anthony-albanese-jakarta-subianto-liberal-coalition-sussan-ley-energy-trade-tariffs-russia-defence-ntwnfb   
350                                                   australia-news/live/2025/jun/26/defence-spending-nsw-queensland-parliament-banks-predict-rate-cuts-in-july-ntwnfb   
399                                                     australia-news/2022/jan/18/five-great-reads-an-american-downton-parental-impostor-syndrome-and-a-70s-must-watch   
422                      australia-news/live/2024/jul/11/australia-news-live-fire-factory-melbourne-alice-springs-curfew-violence-extended-northern-territory-nt-police   
448                  australia-news/live/2024/may/08/news-live-russian-hacker-cybercrime-military-south-china-sea-inflation-interest-rates-cost-of-living-health-vaping   
457   australia-news/live/2023/oct/17/australia-politics-live-question-time-israel-palestine-hamas-anthony-albanese-peter-dutton-indigenous-treaty-truth-cost-of-living   
462                                                                           world/2021/jan/17/first-fruits-of-vaccine-rollout-should-be-seen-in-weeks-experts-predict   
467                                                                                    world/2020/dec/11/oxford-covid-vaccine-to-be-combined-with-sputnik-jab-for-trial   
470                                                                                           world/2021/jan/07/boris-johnson-announces-national-vaccine-booking-scheme   
487                                                  australia-news/live/2023/feb/19/australia-news-live-nsw-election-dominic-perrottet-linda-reynolds-brittany-higgins   
498                                                                                                             commentisfree/2021/jan/08/covid-wave-healthcare-workers   
506                   australia-news/live/2023/mar/17/fifa-saudi-sponsor-productivity-report-boost-wages-sydney-trains-fortescue-bottle-water-submarines-albanese-aukus   
518                                                                         world/2020/may/07/from-brisbane-to-puerto-rico-how-a-38m-covid-19-test-kit-deal-turned-sour   
528                                                                           world/2020/may/15/churches-consider-ticketing-when-they-reopen-after-lockdown-coronavirus   
568                                        australia-news/live/2022/aug/30/australia-news-chris-dawson-covid-isolation-politics-anthony-albanese-skills-summit-tax-cuts   
619                                                                                        world/2020/apr/20/nhs-frontline-meet-people-risking-lives-tackle-coronavirus   
636                                 world/live/2021/dec/07/co

***
The majority of topics are related to coronavirus and have little in common to do with healthy ageing.
***

In [17]:
guardian_out = guardian_out[guardian_out['topic']!=7]

#### Cleaning topic-6

***
['fridge', 'blender', 'sauce', 'bowl', 'pesto', 'flour', 'espresso', 'chef',<br>
'stove', 'recipe', 'machines', 'marmalade', 'seeds', 'mixer', 'stir', 'dough',<br>
'flavour', 'tsp', 'gas', 'ketchup', 'cooked', 'chilli', 'grill', 'boil', 'recipes',<br>
'baking', 'noodles', 'chopped', 'warranty', 'bake']
***

In [23]:
topic_6 = guardian_out[guardian_out['topic']==6]

In [24]:
topic_6[['id','section','headline','trailText', 'links','tags']]

id  \
175                                                                                    thefilter/2025/jul/11/best-camping-stoves-uk   
187                                    environment/2024/apr/18/ways-keep-leftover-food-fresh-opened-tins-cut-avocados-chefs-experts   
238                    lifeandstyle/2023/dec/13/how-to-maximise-your-fridges-efficiency-dont-trust-the-settings-theyre-aspirational   
247                                                                                             thefilter/2025/jan/07/best-blenders   
251       lifeandstyle/2023/nov/29/stir-fry-recipe-simple-easy-vegetarian-vegan-snow-peas-tempeh-green-beans-chilli-alice-zaslavsky   
265                                                               thefilter/2024/oct/30/best-stand-mixers-kitchen-kneading-whipping   
270                                                                                      thefilter/2024/nov/21/best-coffee-machines   
274                                                                     food/2023/jul/03/should-you-refrigerate-ketchup-store-shelf   
299                                  food/2023/sep/03/masala-zone-piccadilly-circus-a-profoundly-cheering-venture-restaurant-review   
427             food/2021/oct/25/long-hours-gruelling-work-at-gourmet-travellers-restaurant-awards-winner-calls-for-industry-change   
458                                                    food/2020/feb/09/forget-wellness-marmalade-is-the-key-to-a-long-healthy-life   
525                                             lifeandstyle/2021/jul/31/oslo-pram-guy-teenage-vacuum-expert-niche-online-reviewers   
579                                                 lifeandstyle/2020/jun/21/family-chefs-restaurateurs-following-fathers-footsteps   
755                                          lifeandstyle/2018/jan/21/lescargot-london-lets-hope-it-never-changes-restaurant-review   
760                                          travel/2018/feb/02/food-tour-cilento-campania-italy-mediterranean-diet-delia-morinelli   
765                                       lifeandstyle/2018/jul/12/ready-for-your-selfie-why-public-spaces-are-being-insta-designed   
772                                                                 lifeandstyle/2018/feb/16/dan-dan-noodles-recipe-felicity-cloake   
797                                  lifeandstyle/2017/feb/10/vasco-and-piero-pavilion-london-w1-restaurant-review-marina-oloughlin   
807                                                                      lifeandstyle/2017/aug/06/how-britain-fell-for-wetherspoons   
888                                                     lifeandstyle/2016/oct/16/ofm-awards-2016-best-food-personality-jamie-oliver   
897                                                                                 environment/2015/oct/18/eco-guide-to-eating-out   
907                                   lifeandstyle/2016/feb/25/lahori-charga-recipe-cottage-cheese-chapati-sumayya-usmani-residency   
908                    lifeandstyle/2016/may/26/seasonal-recipe-ideas-for-beetroot-leeks-root-vegetables-oliver-rowe-cook-residency   
935                                                       lifeandstyle/2015/dec/17/how-bake-perfect-biscotti-coffee-felicity-cloake   
944   lifeandstyle/2015/dec/17/apple-chutney-recipe-rhubarb-and-ginger-shrub-homemade-preserves-jarring-kylee-newton-cook-residency   
963                                                    lifeandstyle/wordofmouth/2015/jan/26/chef-neil-rankin-why-i-hate-food-trends   
975                                                    tv-and-radio/2015/mar/19/bbc-food-television-diet-giles-coren-kew-on-a-plate   
987                                        lifeandstyle/2016/feb/21/los-angeles-fast-food-revolution-locol-roy-cho-daniel-patterson   
992                                                            lifeandstyle/2015/may/03/percy-and-founders-london-restaurant-review   
1012                                                    sustainable-business/20

***
The topics are mostly related to food, so I will get rid of them.
***

In [18]:
guardian_out = guardian_out[guardian_out['topic']!=6]

#### Cleaning topic-5

***
['divorce', 'romantic', 'marry', 'sonya', 'jackie', 'maureen', 'romance',<br>
'trish', 'toby', 'gay', 'phil', 'divorced', 'marriages', 'emmy', 'mike',<br>
'getting married', 'ted', 'weddings', 'pamela', 'pete', 'vegas', 'brad',<br>
'pregnant', 'kathleen', 'fiona', 'boyfriend', 'adrian', 'dating', 'friendships', 'christ']
***

In [27]:
topic_5 = guardian_out[guardian_out['topic']==5]

In [28]:
topic_5[['id','section','headline','trailText', 'links','tags']]

,id,section,headline,trailText,links,tags
167,lifeandstyle/2024/jan/09/foot-fetish-wife-no-longer-let-me-near-her-feet,Life and style,"Since I admitted to a foot fetish, my wife will no longer let me near her feet",It felt like a way to connect but now she has withdrawn from me. I feel as if sex is always on her terms,mailto:private.lives@theguardian.com; https://www.theguardian.com/letters-terms,Life and style; Sex
174,wellness/2024/feb/02/prioritize-friend-relationships-loneliness-health,Wellness,"‘Face-to-face, hip-to-hip’ friendships help us live longer – so let’s prioritize them","Loneliness is a health epidemic, but friendships can lower blood pressure and cardiovascular reactivity – and make us happier",https://www.insider.com/best-girlfriends-build-stunning-home-where-they-will-retire-together-2019-7; https://www.theguardian.com/wellness/2024/feb/01/dog-etiquette-public-training-covid; https://www.theguardian.com/books/2024/jan/29/is-it-ever-okay-to-ghost-a-friend-its-complicated-gyan-yankovich; https://www.census.gov/library/stories/2023/07/marriage-divorce-rates.html; https://www.psychologytoday.com/us/articles/201009/revenge-the-introvert; https://www.vice.com/en/article/pg7njy/enjoy-your-friends-before-you-lose-them-all-at-age-25; https://www.cdc.gov/aging/publications/features/lonely-older-adults.html; https://journals.sagepub.com/doi/10.1177/1745691614568352; https://journals.plos.org/plosmedicine/article?id=10.1371/journal.pmed.1000316; https://www.hhs.gov/sites/default/files/surgeon-general-social-connection-advisory.pdf; https://www.sciencedirect.com/science/article/pii/S235282732200310X?via%3Dihub; https://www.thefriendshipblog.com/about-the-friendship-doctor/the-friendship-doctor/; https://interactive.guim.co.uk/uploader/embed/2023/11/archive-zip/giv-13425SACijDhlvmi7/; https://www.bluezones.com/2018/08/moai-this-tradition-is-why-okinawan-people-live-longer-better/; https://www.bluezones.com/about/history/; https://academic.oup.com/abm/article/33/3/278/4569368; https://academic.oup.com/abm/article/33/3/278/4569368; https://www.sciencedirect.com/science/article/abs/pii/S002210310800070X?via%3Dihub; https://www.healthline.com/health/relationships/chosen-family#examples; https://www.theguardian.com/wellness/2024/jan/30/communal-living-parenting-isolation-babies-families,Well actually; Friendship; Mental health; Health; Life and style; Relationships; Ageing; Society
208,commentisfree/2023/sep/11/the-secrets-of-a-happy-relationship-sleep-luck-and-takeaways,Opinion,"The secrets of a happy relationship? Sleep, luck and takeaways","A survey asking couples why they stayed together proves the uselessness of people’s opinions. It’s like asking centenarians the secret of a long life, writes Emma Beddington",https://www.theguardian.com/tv-and-radio/2023/sep/07/love-death-review-elizabeth-olsen-is-mesmerising-as-an-axe-murderer; https://www.guinnessworldrecords.com/news/2020/10/the-worlds-oldest-people-and-their-secrets-to-a-long-life-632895,Relationships; Life and style
279,lifeandstyle/2023/jul/01/this-is-how-we-do-it-as-weve-aged-weve-adapted-we-now-love-morning-sex-,Life and style,"This is how we do it: ‘As we’ve aged, we’ve adapted: we now love morning sex’","Clyde and Audrey start each day with a spooning session, and allow each other fantasies",mailto:sexlives@theguardian.com,Life and style; Sex; Relationships
280,books/ng-interactive/2024/may/07/anne-lamott-author-interview,Wellness,"Anne Lamott on love, sobriety and reaching 70: ‘All I’ve learned, I’ve learned because the abyss swallowed me’","The American author’s 20th book reads as a series of parables on grace. In an interview, she reflects on a life she thought she wouldn’t get to live",https://www.nytimes.com/2024/04/21/books/review/anne-lamott-somehow.html; https://www.penguinrandomhouse.com/books/734582/somehow-by-anne-lamott/; https://www.kirkusreviews.com/book-reviews/anne-lamott/bird-by-bird/; https://www.washingtonpost.com/opinions/2023/12/20/aging-fri

***
Somewhat related articles:<br>
wellness/2024/feb/02/prioritize-friend-relationships-loneliness-health - 174<br>
Everything else should be deleted.
***

In [19]:
guardian_out = guardian_out[(guardian_out['topic'] != 5) | (guardian_out.index == 174)]

#### Cleaning topic-4

***
['elephants', 'conservation', 'bat', 'bats', 'ivory', 'whales', 'velvet',<br>
'populations', 'nest', 'whale', 'ants', 'attenborough', 'temperate', 'habitat',<br>
'endangered', 'david attenborough', 'crabs', 'mouse', 'breeding', 'arctic', 'mimicry',<br>
'crab', 'turtles', 'tunnel', 'predators', 'dolphins', 'nestling', 'chernobyl', 'wolves', 'national park']
***

In [36]:
topic_4 = guardian_out[guardian_out['topic']==4]

In [44]:
show(topic_4[['headline', 'trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
Articles somewhat related to research:<br>
African killifish may hold key to stopping ageing in humans	 - 534<br>
Other articles have little to do with healthy ageing of humans, mainly focusing on the ageing of animals.<br>
***

In [20]:
guardian_out = guardian_out[(guardian_out['topic'] != 4) | (guardian_out.index == 534)]

#### Cleaning topic-3

***
['fasting', 'obesity', 'diets', 'calories', 'mushrooms', 'obese', 'longo',<br>
'weight loss', 'overweight', 'intake', 'junk', 'junk food', 'vitamin', 'tofu',<br>
'resveratrol', 'dietary', 'patel', 'yoghurt', 'type diabetes', 'hockney',<br>
'eating disorders', 'dairy', 'vitamins', 'nutrients', 'calorie', 'glp', 'dieting',<br>
'compounds', 'creme', 'weight gain']
***

In [48]:
topic_3 = guardian_out[guardian_out['topic']==3]

In [50]:
show(topic_3[['section','headline','trailText','tags']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
Unrelated topics:
Who needs quinoa? 17 overlooked and affordable superfoods, from peas and potatoes to popcorn and even sugar	- 49<br>
All these health scares are making me ill. I need someone to tell me croissants are good for you	<br>
Microdosing: how ‘off-label’ use of weight loss jabs is spreading from US to UK	<br>
Ultra-processed food? Forever chemicals? Declining birth rates? What’s behind rising cancer in the under-50s?<br>	
Keep taking the vitamin D tablets	- 621<br>
Science and mince pies don’t make a good Christmas cocktail	 - 658<br>
The Mediterranean diet is in retreat even in Italy. What now for the foodies’ ideal?	- 697<br>
John Yudkin was a visionary on the harm caused by sugar	- 875<br>
It’s nonsense that the poor can’t cook or eat well cheaply	- 895<br>
Shellshock! Cadbury comes clean on Creme Egg chocolate change	- 971<br>
Do 'superfoods' really exist?	- 1008<br>
Fat China: how are policymakers tackling rising obesity?	 - 1037<br>
David Hockney should stick to painting	 - 1125<br>

Slightly-related topics:
The actor and influencer has been called out for glorifying restricted eating. What is it with the rich and their weird ideas about wellness?	- 197<br>
Intermittent fasting: what is it, how does it work – and is it right for you?	- 219<br>
The truth about protein: how to get enough – at every age	- 273<br>
Scientists hope weight-loss drugs could treat addiction and dementia	- 283<br>
18 foods to boost your health – and the planet’s	 - 319<br>
Environmental toxins are worsening obesity pandemic, say scientists	- 424<br>
Does turmeric’s reputation translate into real health benefits?	 - 426<br>
Forget the fries – here’s how we should really get our five a day	 - 443<br>
Extreme fasting: how Silicon Valley is rebranding eating disorders	 - 459<br>
Dopamine fasting: why Silicon Valley is trying to avoid all forms of stimulation	 - 464<br>
British women live shorter lives than most other Europeans	 - 570<br>
US researchers seek to end carbs v fat 'diet wars'	- 627<br>
Modern tribes: the diet guru	- 654<br>
Fit in my 40s: ‘I hadn’t realised that fermentation is so vital to the gut’	 - 690<br>
Hidden, destructive costs of a vegan diet	- 713<br>
Raw deal: is there really any benefit to an uncooked diet?	 - 725<br>
The male diet boom: why men are tackling their midlife obesity crisis	- 745<br>
Michael Mosley: ‘No male in my family has made it beyond 72’	- 901<br>
My hot tips for parents with a fat kid: feed them fun, kindness and dignity	 - 986<br>
Fasting facts: is the 5:2 diet too good to be true?	- 1007<br>
11 ways to eat more healthily – without ruining your life	- 1022<br>
This column will change your life: clearer costs, better decisions	 - 1094<br>
Sugar is bad for us and we should all stop eating it – right?	- 1121<br>
Keep yourself healthy. Way better than asking a doctor like me to do it for you	- 1145<br>
Oh dear. I'm eating myself to death	- 1158<br>
Public health: saving lives and spending less	 - 1185<br>
High-heel orgasms? Colon cleanses? Celebrity 'science' is a bad joke	- 1190<br>
Have we had our fill of water?	 - 1275<br>
Consider tofu	- 1286<br>

Related-topics:
Life expectancy growth stalls across Europe as England sees sharpest decline, say researchers	 - 42<br>
Becoming obese under age of 30 ‘raises risk of early death by at least 75%’	- 55<br>
Daily multivitamins do not help people live longer, major study finds	- 133<br>
Mushroom magic: why the latest health fad might be on to something	- 492<br>
The Silicon Valley execs who don't eat for days: 'It's not dieting, it's biohacking'	 - 546<br>
Red and processed meat can shorten life, say scientists	- 601<br>
Frequent spicy meals linked to human longevity	- 727<br>
Get off the treadmill: the art of living well in the age of plenty	- 751<br>
Long life? I think we’ve cracked it	- 936<br>
Diets high in meat, eggs and dairy could be as harmful to health as smoking	- 1035<br>
Life is now. Don't let the professional anti-smoking brigade ruin it	- 1043<br>
The secret of the Mediterranean diet? There is no secret	 - 1046<br>
Calorie restriction doesn't slow ageing, monkey study suggests	 - 1079<br>
TV review: Horizon: Eat, Fast and Live Longer; Britain's High Street Gamble	 - 1171<br>
Consider yoghurt	- 1264<br>
Can resveratrol – the 'wonder chemical' in red wine – live up to the hype?	 - 1064<br>
‘I was scared to even eat the vegetables in my fridge’: the eating disorder that focuses on food purity	- 84<br>
From cold showers to hot tomatoes: 10 of Michael Mosley’s top health tips	- 183<br>
<br>
A lot of these topics are related to our research, so we leave them.
***

#### Cleaning topic-2

***
['makeup', 'serum', 'skincare', 'anti ageing', 'glow', 'nail', 'gel',<br>
'concealer', 'lashes', 'lip', '30ml', 'creams', 'facial', 'moisturiser',<br>
'balm', 'vitamin', 'wrinkles', 'brow', 'lipstick', '50ml', 'sunscreen',<br>
'retinol', 'mascara', 'cleansing', 'botox', 'powder', 'moisturising', 'polish', 'cleanser', 'scent']
***

In [6]:
topic_2 = guardian_out[guardian_out['topic']==2]

In [8]:
show(topic_2[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
These topics mostly cover the beauty part of the longevity, i.e. anti-ageing serums,perfumes, so we disregard these.
***

In [21]:
guardian_out = guardian_out[guardian_out['topic'] != 2]

#### Cleaning topic-1

***
['shirt', 'clothing', 'jeans', 'leather', 'wardrobe', 'boots',<br>
'fabric', 'wool', 'trousers', 'vogue', 'dresses', 'designs',<br>
'designers', 'shirts', 'jacket', 'wore', 'cashmere', 'burton',<br>
'oscar', 'socks', 'secondhand', 'mulberry', 'dolly', 'skinny',<br>
'fabrics', 'parton', 'skirt', 'garment', 'catwalk', 'garments']
***

In [10]:
topic_1 = guardian_out[guardian_out['topic']==1]

In [11]:
show(topic_1[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
These topics mostly concern fashion and celebrity ageing, so they are not related to our research as much.
***

In [22]:
guardian_out = guardian_out[guardian_out['topic']!=1]

#### Reviewing guardian_out

In [44]:
guardian_out.to_csv(GUARDIAN_DATA_DIR / 'guardian_cleared2.csv', index=False)

<!-- structured-notebook -->
## Round 3 Overview
After saving `guardian_cleared2.csv`, the notebook repeats the same review pattern on the newly cleaned subset to see whether the topic structure becomes more coherent.


### BERTopic Cleaning round 3

In [3]:
guardian_out = pd.read_csv(GUARDIAN_DATA_DIR / 'guardian_cleared2.csv')

In [5]:
guardian_out = guardian_out.drop(columns=['topic', 'topic_prob','topic_keywords'])

In [6]:
model, topic_info, topic_out = analyze_bertopic(guardian_out, v_min_df=1, v_max_df=0.8,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

2025-07-27 14:21:29,329 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████████████████████████████| 22/22 [02:55<00:00,  7.96s/it]
2025-07-27 14:24:24,591 - BERTopic - Embedding - Completed ✓
2025-07-27 14:24:24,592 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-07-27 14:24:29,593 - BERTopic - Dimensionality - Completed ✓
2025-07-27 14:24:29,594 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-07-27 14:24:29,614 - BERTopic - Cluster - Completed ✓
2025-07-27 14:24:29,614 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-07-27 14:24:30,555 - BERTopic - Representation - Completed ✓
2025-07-27 14:24:30,555 - BERTopic - Topic reduction - Reducing number of topics
2025-07-27 14:24:30,559 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-07-27 14:24:31,542 - BERTopic - Representation - Completed ✓
2025-07-27 14:24:31,543 - BERTopic 

In [7]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,119,-1_pfas_chemicals_company_life expectancy,"[pfas, chemicals, company, life expectancy, ex...",[People in their 50s can be prone to a “viciou...
1,0,182,0_cells_mice_scientists_biological,"[cells, mice, scientists, biological, drug, ol...",[The science of extending life is a subject of...
2,1,153,1_pension_retirement_expectancy_life expectancy,"[pension, retirement, expectancy, life expecta...","[In a country so grossly unequal, nothing is f..."
3,2,65,2_fitness_muscle_yoga_gym,"[fitness, muscle, yoga, gym, workout, strength...","[Like buying hats or undergarments, staying he..."
4,3,56,3_fasting_protein_diet_eating,"[fasting, protein, diet, eating, fat, eat, sug...",[Eating is so last season; these days all the ...
5,4,43,4_nhs_mps_amendment_secretary,"[nhs, mps, amendment, secretary, hunt, politic...","[“Call this a cure for NHS, Sir Keir?” splashe..."
6,5,27,5_life expectancy_expectancy_blackpool_glasgow,"[life expectancy, expectancy, blackpool, glasg...","[In Glasgow's glittering West End, known as th..."
7,6,14,6_dementia_brain_music_memory,"[dementia, brain, music, memory, cognitive, br...","[Brain health is key to successful ageing, and..."
8,7,13,7_abortion_coverage_rights_insurance,"[abortion, coverage, rights, insurance, regan,...",[Jad Adams’s new global history of women’s enf...
9,8,11,8_christmas_tree_trees_winter,"[christmas, tree, trees, winter, calendar, adv...",[It’s the Christmas present that nobody wanted...


In [10]:
full_keyword_getter(topic_out)

-1.0 → ['pfas', 'chemicals', 'company', 'life expectancy', 'expectancy', 'coffee', 'calment', 'ai', 'nhs', 'spa', 'diseases', 'brain', 'zak', 'dementia', 'diet', 'treatments', 'tea', 'conditions', 'biobank', 'skin', 'bacteria', 'environmental', 'scientific', 'jennifer', 'genetic', 'claims', '23andme', 'linked', 'animals', 'epa']
0.0 → ['cells', 'mice', 'scientists', 'biological', 'drug', 'oldest', 'diet', 'diseases', 'cell', 'life expectancy', 'expectancy', 'process', 'centenarians', 'hair', 'anti ageing', 'drugs', 'sleep', 'senescent', 'brain', 'company', 'johnson', 'senescent cells', 'stem', 'genetic', 'eat', 'plasma', 'test', 'genes', 'trial', 'scientific']
1.0 → ['pension', 'retirement', 'expectancy', 'life expectancy', 'state pension', 'pensions', 'housing', 'workers', 'jobs', 'father', 'pension age', 'generation', 'grandchildren', 'active', 'asked', 'paid', 'daughter', 'career', 'married', 'young people', 'loneliness', 'employers', 'ageism', 'insurance', 'social care', 'pensioner

#### Cleaning topic-8

***
['christmas', 'tree', 'trees', 'winter', 'calendar', 'advent',<br>
'calendars', 'christmas tree', 'artificial', 'advent calendar',<br>
'christmas trees', 'advent calendars', 'garden', 'new year',<br>
'festive', 'beauty', 'grown', '6ft', 'zoe', 'wrapping', 'snow',<br>
'carbon', 'pot', 'dark', 'season', 'buying', 'tradition', 'hibernation',<br>
'december', 'sustainable']
***

In [13]:
guardian_out = topic_out.copy()

In [ ]:
topic_8 = guardian_out[guardian_out['topic']==8]
show(topic_8[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

***
We may delete topic 8 since it is nowhere near related to healthy ageing.
***

In [16]:
guardian_out = guardian_out[guardian_out['topic']!=8]

#### Cleaning topic-7

***
['abortion', 'coverage', 'rights', 'insurance', 'regan', 'gender',<br>
'political', 'violence', 'pregnancy', 'male', 'female', 'vote',<br>
'feminist', 'adams', 'substance', 'misuse', 'abortion rights',<br>
'contraception', 'sexual', 'pregnancies', 'africa', 'supreme',<br>
'patent', 'abuse', 'pregnant', 'maternal', 'abortions', 'sex',<br>
'court', 'domestic']
***

In [17]:
topic_7 = guardian_out[guardian_out['topic']==7]
show(topic_7[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
These topics are also unrelated to our research.
***

In [18]:
guardian_out = guardian_out[guardian_out['topic']!=7]

#### Cleaning topic-6

***
['dementia', 'brain', 'music', 'memory', 'cognitive', 'brain health',<br>
 'alzheimer', 'onset', 'cognitive decline', 'onset dementia', 'people dementia',<br>
 'elo', 'early onset', 'grandmother', 'socioeconomic', 'exercises', 'sleep',<br>
 'lancet', 'risk factors', 'link', 'letter', 'fish', 'pet', 'sharon', 'habits',<br>
 'healthy lifestyle', 'caregiving', 'games', 'therapy', 'physical exercise']
***

In [19]:
topic_6 = guardian_out[guardian_out['topic']==6]
show(topic_6[['section','headline','trailText','tags','links']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
For now, let us take them out, then we will discuss it together.
***

In [20]:
guardian_out = guardian_out[guardian_out['topic']!=6]

#### Cleaning topic-5

***
['life expectancy', 'expectancy', 'blackpool', 'glasgow', 'young people',<br>
 'inequalities', 'expect live', 'nations', 'mortality', 'australia', 'marmot',<br>
 'kensington', 'kensington chelsea', 'nhs', 'inequality', 'good health',<br>
 'males', 'chelsea', 'expectancies', 'housing', 'male', 'life expectancies',<br>
 'poor health', 'healthy life', 'deprived', 'poorest', 'ons', 'office', 'scotland',<br>
 'statistics']
***

In [22]:
topic_5 = guardian_out[guardian_out['topic']==5]
show(topic_5[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


In [26]:
keep_ids = [23,102,154,211,241,258,293,330,344,414,506,572,622,638,647]

other_topics = guardian_out[guardian_out['topic'] != 5]

topic_5_keep = topic_5[topic_5.index.isin(keep_ids)]

guardian_out = pd.concat([other_topics, topic_5_keep]).sort_index().copy()

#### Cleaning topic-4

***
['nhs', 'mps', 'amendment', 'secretary', 'hunt', 'political',<br>
'northern', 'northern ireland', 'staff', 'vote', 'reform',<br>
'tory', 'scottish national', 'national party', 'scottish',<br>
'health service', 'gps', 'social care', 'queen speech', 'queen',<br>
'scheme', 'insurance', 'brexit', 'patient', 'decision', 'ministers',<br>
'cameron', 'creasy', 'jeremy', 'junior']
***

In [30]:
topic_4 = guardian_out[guardian_out['topic']==4]
show(topic_4[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
None of the listed articles provide direct data or focused analysis on longevity / life‑expectancy gaps; they mainly concern NHS politics, strikes, or broad opinion columns.
***

In [31]:
guardian_out = guardian_out[guardian_out['topic']!=4]

#### Cleaning topic-3

***
['fasting', 'protein', 'diet', 'eating', 'fat', 'eat',<br>
'sugar', 'foods', 'curcumin', 'salt', 'meat', 'processed',<br>
'mushrooms', 'fast', 'china', 'calories', 'longo', 'nutrition',<br>
'obese', 'diets', 'overweight', 'weight loss', 'egg', 'loss',<br>
'type', 'tofu', 'drugs', 'milk', 'meals', 'patel']
***

In [32]:
topic_3 = guardian_out[guardian_out['topic']==3]
show(topic_3[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


In [33]:
keep_ids = [41, 86, 135, 226, 228, 245, 262, 283, 314, 392, 520, 534, 607, 613]

other_topics = guardian_out[guardian_out['topic'] != 3]

topic_3_keep = topic_3[topic_3.index.isin(keep_ids)]

guardian_out = pd.concat([other_topics, topic_3_keep]).sort_index().copy()

#### Cleaning topic-2

***
['fitness', 'muscle', 'yoga', 'gym', 'workout', 'strength',<br>
'muscles', 'leg', 'balance', 'intensity', 'sport', 'lower',<br>
'physical activity', 'workouts', 'creatine', 'exercises',<br>
'dancers', 'push', 'injury', 'esports', 'laughter', 'legs',<br>
'feet', 'classes', 'pilates', 'wall', 'lifting', 'runners',<br>
'foot', 'marathon']
***

In [34]:
topic_2 = guardian_out[guardian_out['topic']==2]
show(topic_2[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


In [35]:
keep_ids = [29, 57, 85, 87, 91, 116, 205, 217, 227, 356, 393, 603]

other_topics = guardian_out[guardian_out['topic'] != 2]

topic_2_keep = topic_2[topic_2.index.isin(keep_ids)]

guardian_out = pd.concat([other_topics, topic_2_keep]).sort_index().copy()

#### Cleaning topic-1

***
['pension', 'retirement', 'expectancy', 'life expectancy', 'state pension',<br>
'pensions', 'housing', 'workers', 'jobs', 'father', 'pension age', 'generation',<br>
'grandchildren', 'active', 'asked', 'paid', 'daughter', 'career', 'married',<br>
'young people', 'loneliness', 'employers', 'ageism', 'insurance', 'social care',<br>
'pensioners', 'sex', 'retirement age', 'mum', 'homes']
***

In [36]:
topic_1 = guardian_out[guardian_out['topic']==1]
show(topic_1[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


In [38]:
keep_ids = [
    54, 94, 111, 121, 151, 167, 207, 236, 269,
    284, 291, 301, 321, 394, 410, 423, 426, 438,
    456, 475, 482, 484, 489, 491, 498, 500, 516,
    526, 529, 537, 550, 554, 580, 590, 592, 593,
    606, 611, 640, 641, 652, 665
]

other_topics = guardian_out[guardian_out['topic'] != 1]

topic_1_keep = topic_1[topic_1.index.isin(keep_ids)]

guardian_out = pd.concat([other_topics, topic_1_keep]).sort_index().copy()

#### Cleaning topic-0

In [39]:
topic_0 = guardian_out[guardian_out['topic']==0]
show(topic_0[['section','headline','trailText']], scrollY="500px", scrollCollapse=True)

Loading ITables v2.4.4 from the internet... (need help?)


***
The topic-0 is too bulky for manual cleaning. I will have to perform another round of cleaning here. 
***

In [45]:
guardian_out.to_csv(GUARDIAN_DATA_DIR / 'guardian_cleared3.csv', index=False)

<!-- structured-notebook -->
## Round 4 Overview
This final pass is mostly a check on whether repeated cleaning has stabilized the topic landscape. It ends with a short summary and a fresh BERTopic run on the broader Guardian corpus.


### BERTopic Cleaning round 4

In [3]:
guardian_out = pd.read_csv(GUARDIAN_DATA_DIR / 'guardian_cleared3.csv')

In [5]:
guardian_out = guardian_out.drop(columns=['topic','topic_prob','topic_keywords'])

model, topic_info, guardian_out = analyze_bertopic(guardian_out, v_min_df=1, v_max_df=0.6,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

In [8]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,111,-1_pfas_chemicals_products_genetic,"[pfas, chemicals, products, genetic, calment, ...","[If time makes fools of us all, you couldn’t b..."
1,0,85,0_mice_drug_hair_anti ageing,"[mice, drug, hair, anti ageing, senescent, pla...","[In the summer of 2019, months before the word..."
2,1,79,1_retirement_pension_housing_nhs,"[retirement, pension, housing, nhs, pensions, ...",[Increased longevity with better health and we...
3,2,66,2_japan_oldest_japanese_world oldest,"[japan, oldest, japanese, world oldest, south,...","[The year Misao Okawa was born, Queen Victoria..."
4,3,19,3_leg_intensity_training_strength,"[leg, intensity, training, strength, minutes, ...","[Slow down, that used to be the mantra for mid..."
5,4,13,4_optimism_neuroticism_optimistic_happiness,"[optimism, neuroticism, optimistic, happiness,...",[People who have a rosy outlook on the world m...
6,5,11,5_protein_longo_nezis_vegan,"[protein, longo, nezis, vegan, woo, libin, cal...","[A diet rich in meat, eggs, milk and cheese co..."


In [9]:
full_keyword_getter(guardian_out)

-1.0 → ['pfas', 'chemicals', 'products', 'genetic', 'calment', 'curcumin', 'spa', 'ai', 'treatments', 'biobank', 'zak', 'companies', 'technology', 'love', 'environmental', 'house', 'air', 'bacteria', '23andme', 'jennifer', 'questions', 'app', 'epa', 'stem', 'travel', 'insurance', '101', 'wellness', 'ink', 'married']
0.0 → ['mice', 'drug', 'hair', 'anti ageing', 'senescent', 'plasma', 'senescent cells', 'johnson', 'animals', 'trials', 'products', 'treatments', 'silicon valley', 'silicon', 'wyss', 'coray', 'wyss coray', 'dogs', 'nucalm', 'mushrooms', 'dna', 'technology', 'space', 'rapamycin', 'product', 'companies', 'psychedelics', 'proteins', 'organs', 'pig']
1.0 → ['retirement', 'pension', 'housing', 'nhs', 'pensions', 'social care', 'income', 'services', 'dementia', 'inequalities', 'residents', 'state pension', 'workers', 'poverty', 'private', 'policy', 'poor health', 'spending', 'homes', 'healthy life', 'decade', 'sector', 'blackpool', 'pay', 'dogs', 'cost', '89', 'cuts', 'expectanci

# Summary

**Results:** <br>
1. The data cleaning pipeline was performed in 4 rounds, narrowing down the number of area of topics and focusing on longevity.
2. The data for every cleaning round was saved in csv format in case the process has to be followed through.
3. The resulted dataset contains 385 articles.
<br>**Found keywords:**<br>
1. 5:2 diet - living longer by fasting
2. rapamycin - Studied extensively for its potential anti-ageing effects via mTOR inhibition.
3. senescent cells, senescent plasma - cells that make people age
4. curcumin - has some side effects that help with ageing
5. myocine - referred to as hope molecules within news
6. supercentenarian - a person who lived to be 110 years older or longer
7. Blue Zones - regions around the world where people live significantly longer and healthier lives, often reaching the age of 100 and beyond

In [13]:
guardian = pd.read_csv(GUARDIAN_DATA_DIR / 'guardian_data_2010_2025_links.csv')

In [18]:
guardian = clean_tags(guardian,['ageing'])

Kept 379 of 616 articles related to ageing.


In [15]:
model, topic_info, guardian_out = analyze_bertopic(guardian, v_min_df=1, v_max_df=0.6,text_col="bodyText",top_n_words=30,
                                     save_csv=None)

2025-08-13 09:25:15,924 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████████████████████████████| 21/21 [03:47<00:00, 10.85s/it]
2025-08-13 09:29:03,899 - BERTopic - Embedding - Completed ✓
2025-08-13 09:29:03,900 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-08-13 09:29:04,370 - BERTopic - Dimensionality - Completed ✓
2025-08-13 09:29:04,370 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-08-13 09:29:04,418 - BERTopic - Cluster - Completed ✓
2025-08-13 09:29:04,418 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-08-13 09:29:05,530 - BERTopic - Representation - Completed ✓
2025-08-13 09:29:05,531 - BERTopic - Topic reduction - Reducing number of topics
2025-08-13 09:29:05,534 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-08-13 09:29:06,598 - BERTopic - Representation - Completed ✓
2025-08-13 09:29:06,599 - BERTopic 

In [16]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,152,-1_coronavirus_covid 19_snow_stonehenge,"[coronavirus, covid 19, snow, stonehenge, vacc...",[A team of American scientists report they’ve ...
1,0,130,0_phd_mice_neuroscience_usa,"[phd, mice, neuroscience, usa, phd professor, ...",[In a lab just south of San Francisco I am loo...
2,1,107,1_pension_pelevin_state pension_pensions,"[pension, pelevin, state pension, pensions, ru...","[For the first time in the UK, there are more ..."
3,2,64,2_fitness_yoga_physical activity_dementia,"[fitness, yoga, physical activity, dementia, w...","[With new exercises invented every week, new g..."
4,3,42,3_diet_fasting_blender_curcumin,"[diet, fasting, blender, curcumin, mushrooms, ...","[A diet rich in meat, eggs, milk and cheese co..."
5,4,37,4_star wars_sites_films_ai,"[star wars, sites, films, ai, movie, potter, h...",[The winner The jury may be out on the creativ...
6,5,29,5_sellafield_concrete_solar_batteries,"[sellafield, concrete, solar, batteries, carbo...",[1. Printable solar panels Unlike most solar p...
7,6,25,6_boots_anti ageing_boot_products,"[boots, anti ageing, boot, products, serum, su...",[One cold and wet day in what passes for sprin...
8,7,21,7_vocal_paglin_brilla_adele,"[vocal, paglin, brilla, adele, cords, brilla p...",[There’s a Homeric hymn that sings of a mortal...
9,8,19,8_tropical_temperate_wildlife_whales,"[tropical, temperate, wildlife, whales, songbi...","[She was born during the reign of James I, was..."
